# Mini-GPT on Danish 🇩🇰
Build a GPT **from scratch**, a character-level transformer just like the one inside ChatGPT, only about 1,000,000× smaller. Then train it on **your own Danish notes** and watch it go from random symbols to convincing *nonsense Danish legalese*.

Guide note: [[Mini-GPT on Danish]] · background: [[Transformers]], [[Attention Mechanism]], [[LLMs Overview]]

**The corpus** is every Markdown note in `9-semester/IT-lov/` (≈ 100k characters today), plus any `.txt` files you drop into `data/my_text/` (song lyrics, a book, your own writing). It grows automatically as you add lectures. Training takes ≈ 2 min on a laptop CPU.

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import glob, math, pathlib, re, time, numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

VAULT = pathlib.Path("../..")                                  # this notebook lives in Machine Learning/22-Personal-Projects/
SOURCES = [VAULT / "9-semester/IT-lov"]                         # add more folders with Danish notes here
EXTRA = pathlib.Path("data/my_text"); EXTRA.mkdir(parents=True, exist_ok=True)   # drop .txt files here

def clean_markdown(s):
    """Strip Obsidian/Markdown syntax so the model learns Danish, not formatting."""
    s = re.sub(r"^---\n.*?\n---\n", "", s, flags=re.S)               # frontmatter
    s = re.sub(r"\[\[([^\]|]+\|)?([^\]]+)\]\]", r"\2", s)           # [[link|alias]] → alias
    s = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", s)                  # [text](url) → text
    s = re.sub(r"\[!\w+\]-?", "", s); s = re.sub(r"[#*>`|_]+", "", s)
    s = re.sub(r"-{3,}", "", s); s = re.sub(r"[ \t]+", " ", s); s = re.sub(r"\n\s*\n+", "\n\n", s)
    return s.strip()

files = sorted(f for src in SOURCES for f in src.rglob("*.md")) + sorted(EXTRA.glob("*.txt"))
text = "\n\n".join(clean_markdown(f.read_text(encoding="utf-8")) if f.suffix == ".md" else f.read_text(encoding="utf-8") for f in files)
print(f"{len(files)} files, {len(text):,} characters")
if len(text) < 20_000: print("⚠️  Very little text – the model will mostly memorise. Add .txt files to data/my_text/.")
print(text[:400])

## 1. A character tokenizer
GPT-4 uses ~100k sub-word tokens; we use single **characters**: no external library, and æ, ø, å come for free. Return the sorted list of unique characters plus `encode` (string → list of ints) and `decode` (list of ints → string).

In [ ]:
def make_tokenizer(text):
    # TODO 1: chars = sorted unique characters; encode/decode map between strings and integer ids
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_tok():
    chars, enc, dec = make_tokenizer("bøf og blå bær")
    return chars == sorted(set("bøf og blå bær")) and dec(enc("blå bøf")) == "blå bøf" and enc("b")[0] == chars.index("b")
check("tokenizer", _t_tok, "chars must be sorted unique characters; decode(encode(s)) must give s back.")

In [ ]:
if ready("tokenizer"):
    chars, encode, decode = make_tokenizer(text)
    V = len(chars)
    data = torch.tensor(encode(text))
    n = int(0.9 * len(data)); train_data, val_data = data[:n], data[n:]      # last 10 % = validation
    print(f"vocabulary: {V} characters  →  {''.join(chars)!r}")

    # The number to beat: a bigram model (next char depends only on the current char), Laplace-smoothed
    counts = torch.ones(V, V).index_put_((train_data[:-1], train_data[1:]), torch.ones(n - 1), accumulate=True)
    P = counts / counts.sum(1, keepdim=True)
    bigram_val = -torch.log(P[val_data[:-1], val_data[1:]]).mean().item()
    print(f"uniform guessing: loss {math.log(V):.3f}   bigram baseline: val loss {bigram_val:.3f}")

## 2. Causal self-attention, the heart of GPT
Each position asks a **query**, every position offers a **key** and a **value**. The weights $\text{softmax}(QK^\top/\sqrt{d})$ decide how much of each value to mix in. The **causal mask** sets the scores for *future* positions to $-\infty$ before the softmax, so position $t$ can only look at $\le t$ (otherwise it could cheat by reading the answer).

`q, k, v` have shape `(B, heads, T, head_size)`. Return `softmax(q kᵀ / √head_size + mask) v`.

In [ ]:
def causal_attention(q, k, v):
    # TODO 2: scaled dot-product attention with a lower-triangular (causal) mask
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_att():
    torch.manual_seed(0); q, k, v = torch.randn(3, 2, 4, 5, 8).unbind(0)
    out = causal_attention(q, k, v)
    ref = F.scaled_dot_product_attention(q, k, v, is_causal=True)          # PyTorch's built-in, for comparison
    k2, v2 = k.clone(), v.clone(); k2[..., 3, :] += 5; v2[..., 3, :] += 5   # change position 3 only
    out2 = causal_attention(q, k2, v2)
    return out.shape == (2, 4, 5, 8) and torch.allclose(out, ref, atol=1e-5) and torch.allclose(out[..., :3, :], out2[..., :3, :])
check("causal attention", _t_att, "Must match F.scaled_dot_product_attention(is_causal=True); earlier positions must not see later ones.")

## 3. The transformer block
Pre-norm GPT-2 style: `x = x + attention(LayerNorm(x))`, then `x = x + MLP(LayerNorm(x))`. The `x +` **residual** paths are what make deep transformers trainable (see [[Paper-to-Code Months]] month 1, ResNet).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.h = n_heads; self.qkv = nn.Linear(d, 3 * d, bias=False); self.proj = nn.Linear(d, d); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = (t.view(B, T, self.h, D // self.h).transpose(1, 2) for t in self.qkv(x).split(D, dim=-1))
        y = causal_attention(q, k, v).transpose(1, 2).reshape(B, T, D)
        return self.drop(self.proj(y))

class Block(nn.Module):
    def __init__(self, d, n_heads, dropout):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, n_heads, dropout)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d), nn.Dropout(dropout))
    def forward(self, x):
        # TODO 3: two residual sub-layers, each with LayerNorm applied BEFORE attn / mlp
        raise NotImplementedError  # TODO 3

class MiniGPT(nn.Module):
    def __init__(self, vocab, block_size=64, d=128, n_heads=4, n_layers=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok = nn.Embedding(vocab, d); self.pos = nn.Embedding(block_size, d)
        self.blocks = nn.Sequential(*[Block(d, n_heads, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d); self.head = nn.Linear(d, vocab)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1]))
        return self.head(self.ln(self.blocks(x)))          # logits (B, T, vocab)

In [ ]:
def _t_block():
    blk = Block(16, 2, 0.0); x = torch.randn(2, 5, 16)
    for lin in (blk.attn.proj, blk.mlp[2]):                # zero the output layers → block must be the identity
        nn.init.zeros_(lin.weight); nn.init.zeros_(lin.bias)
    torch.manual_seed(0); m = MiniGPT(30); logits = m(torch.randint(0, 30, (4, 64)))
    loss0 = F.cross_entropy(logits.view(-1, 30), torch.randint(0, 30, (4 * 64,))).item()
    return torch.allclose(blk(x), x) and logits.shape == (4, 64, 30) and abs(loss0 - math.log(30)) < 0.5
check("transformer block", _t_block, "With zeroed output layers the block must return x unchanged (residuals!); a fresh model's loss should be ≈ ln(vocab).")

## 4. Sampling text
Feed the context, take the logits of the **last** position, divide by `temperature` (low = safe and repetitive, high = wild), optionally keep only the `top_k` most likely characters, sample one, append, repeat. Crop the context to the last `model.block_size` characters.

In [ ]:
@torch.no_grad()
def generate(model, idx, n_new, temperature=1.0, top_k=None):
    # TODO 4: autoregressive sampling loop; idx is (B, T) and grows by one column per step
    raise NotImplementedError  # TODO 4

In [ ]:
def _t_gen():
    torch.manual_seed(0); m = MiniGPT(20, block_size=8).eval(); start = torch.zeros(1, 3, dtype=torch.long)
    out = generate(m, start, 12, top_k=1)                                  # top_k=1 → always the argmax
    greedy = start.clone()
    for _ in range(12):
        greedy = torch.cat([greedy, m(greedy[:, -8:])[:, -1].argmax(-1, keepdim=True)], 1)
    return out.shape == (1, 15) and torch.equal(out, greedy)
check("generate", _t_gen, "Output must have T + n_new columns, and top_k=1 must equal greedy decoding (crop to block_size!).")

## 5. Train it and watch it learn
AdamW, batch 32, context 64 characters, 1,500 steps. Every 250 steps we measure train/val loss and print a sample. We keep the weights with the best **validation** loss (early stopping): with this little text the model eventually starts memorising, and the gap between train and val loss shows you exactly when.

In [ ]:
STEPS, EVAL_EVERY, BATCH, LR = 1500, 250, 32, 1e-3
PROMPT = "Den dataansvarlige "

def get_batch(d, bs, T, gen=None):
    ix = torch.randint(len(d) - T - 1, (bs,), generator=gen)
    return torch.stack([d[i:i + T] for i in ix]), torch.stack([d[i + 1:i + T + 1] for i in ix])

@torch.no_grad()
def estimate(model, d, iters=20):
    model.eval(); g = torch.Generator().manual_seed(0)
    losses = [F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item() for x, y in (get_batch(d, BATCH, model.block_size, g) for _ in range(iters))]
    model.train(); return float(np.mean(losses))

def sample(model, prompt=PROMPT, n=200, temperature=0.8, seed=1):
    torch.manual_seed(seed); model.eval()
    out = decode(generate(model, torch.tensor([encode(prompt)]), n, temperature)[0].tolist()); model.train(); return out

if ready("tokenizer", "causal attention", "transformer block", "generate"):
    torch.manual_seed(0)
    model = MiniGPT(V); opt = torch.optim.AdamW(model.parameters(), lr=LR)
    print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")
    hist, best, t0 = [], (float("inf"), None), time.time()
    for step in range(STEPS + 1):
        if step % EVAL_EVERY == 0:
            tr, va = estimate(model, train_data), estimate(model, val_data); hist.append((step, tr, va))
            if va < best[0]: best = (va, {k: v.clone() for k, v in model.state_dict().items()})
            print(f"step {step:>4}  train {tr:.3f}  val {va:.3f}  ({time.time()-t0:.0f}s)")
            if step in (0, 250, STEPS): print("   ↳", sample(model, n=120).replace("\n", " ⏎ "))
        x, y = get_batch(train_data, BATCH, model.block_size)
        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())
        opt.zero_grad(); loss.backward(); opt.step()
    model.load_state_dict(best[1]); best_val = best[0]
    s, tr, va = zip(*hist)
    plt.plot(s, tr, "o-", label="train"); plt.plot(s, va, "o-", label="validation"); plt.axhline(bigram_val, ls="--", c="gray", label="bigram baseline")
    plt.xlabel("step"); plt.ylabel("cross-entropy (nats/char)"); plt.legend(); plt.show()

In [ ]:
if ready("tokenizer", "causal attention", "transformer block", "generate"):
    check("beats the bigram baseline", lambda: best_val < bigram_val - 0.5,
          f"Best val loss {best_val:.3f} should be at least 0.5 below the bigram's {bigram_val:.3f}.")

## 6. Play with it
Temperature changes everything. And how much is it **copying**? `longest_copy` finds the longest piece of a sample that appears word-for-word in your notes. Short copies (10–20 characters) mean it's *composing*; whole sentences mean it's *memorising*.

In [ ]:
def longest_copy(sample_text, corpus):
    best = ""
    for i in range(len(sample_text)):
        j = i + len(best) + 1
        while j <= len(sample_text) and sample_text[i:j] in corpus:
            best = sample_text[i:j]; j += 1
    return best

if ready("tokenizer", "causal attention", "transformer block", "generate"):
    for temp in (0.5, 0.8, 1.2):
        s = sample(model, "Artikel ", n=250, temperature=temp)
        print(f"--- temperature {temp} ---\n{s}\n   longest verbatim copy from the notes: {longest_copy(s, text)!r}\n")

<details><summary>▶ Solution</summary>

```python
def make_tokenizer(text):
    chars = sorted(set(text))
    stoi = {c: i for i, c in enumerate(chars)}
    encode = lambda s: [stoi[c] for c in s]
    decode = lambda ids: "".join(chars[i] for i in ids)
    return chars, encode, decode
```

```python
def causal_attention(q, k, v):
    hs = q.shape[-1]
    T = q.shape[-2]
    w = q @ k.transpose(-2, -1) / math.sqrt(hs)
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=q.device))
    w = w.masked_fill(~mask, float("-inf"))
    return F.softmax(w, dim=-1) @ v
```

```python
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x
```

```python
@torch.no_grad()
def generate(model, idx, n_new, temperature=1.0, top_k=None):
    for _ in range(n_new):
        logits = model(idx[:, -model.block_size:])[:, -1, :] / temperature
        if top_k is not None:
            kth = torch.topk(logits, top_k).values[:, -1, None]
            logits = logits.masked_fill(logits < kth, float("-inf"))
        nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
        idx = torch.cat([idx, nxt], dim=1)
    return idx
```
</details>

## Make it yours (stretch goals)
- **More text:** drop song lyrics or a Danish book (public domain) into `data/my_text/`. Plot best val loss against corpus size. Data beats model size.
- **Bigger model:** `block_size=128`, `n_layers=6`, `d=192`. Does val loss improve, or does it just memorise faster?
- **Sub-word tokens:** train a small BPE tokenizer (e.g. with the `tokenizers` library). Words like *dataansvarlige* become 2–3 tokens instead of 15 characters.
- **Weight tying:** share `self.tok.weight` with `self.head.weight` (GPT-2 does this). Fewer parameters, often better.
- **Learning-rate schedule:** warm-up + cosine decay, like real LLM training.

Check yourself: why is the *validation* split the **last** 10 % of the text and not random characters? (Hint: neighbouring characters overlap in context windows.)